In [28]:
import time

In [1]:
class TemporaryAPIError(Exception):
    pass

In [2]:
class SimulatedTimeoutError(Exception):
    pass

In [3]:
def mock_llm(claim):
    claim_id = claim["claim_id"]

    if claim_id == "C002":
        raise TemporaryAPIError("503 Service Unavailable")

    if claim_id == "C004":
        raise SimulatedTimeoutError("Request timed out")

    return {
        "claim_id": claim_id,
        "status": "PROCESSED"
    }

In [4]:
claims = [
    {"claim_id": "C001", "description": "Engine overheating"},
    {"claim_id": "C002", "description": "Battery not charging"},
    {"claim_id": "C003", "description": "Transmission noise"},
    {"claim_id": "C004", "description": "Brake system warning"},
    {"claim_id": "C005", "description": "AC not cooling"}
]

In [30]:
def call_with_retry(operation, max_attempts=3):
    # Your implementation
    temp = 0
    n = 1

    while temp < max_attempts:
        try:
            result = operation()
            return result
        except (TemporaryAPIError, SimulatedTimeoutError):
            if temp < max_attempts - 1:
                print(f"Attempt {temp+1} Failed: API not available. Retrying after {n} seconds")
                time.sleep(n)
            else:
                print(f"Attempt {temp+1} Failed: No retries remaining")
                raise
            n *= 2
        temp += 1

In [31]:
def process_claims(claims):
    # Your implementation
    successful, failed = [], []
    for claim in claims:
        try:
            result = call_with_retry(lambda: mock_llm(claim=claim))
            # result = mock_llm(claim=claim)
            successful.append(result)
        except(TemporaryAPIError, SimulatedTimeoutError) as e:
            failed.append(
                {
                    'claim_id': claim['claim_id'], 
                    'error': str(e)
                }
            )

    return successful, failed

In [32]:
successful, failed = process_claims(claims)

for claim in successful:
    print(f'{claim['claim_id']} - {claim['status']}')

for claim in failed:
    print(f'{claim['claim_id']} - {claim['error']}')

Attempt 1 Failed: API not available. Retrying after 1 seconds
Attempt 2 Failed: API not available. Retrying after 2 seconds
Attempt 3 Failed: No retries remaining
Attempt 1 Failed: API not available. Retrying after 1 seconds
Attempt 2 Failed: API not available. Retrying after 2 seconds
Attempt 3 Failed: No retries remaining
C001 - PROCESSED
C003 - PROCESSED
C005 - PROCESSED
C002 - 503 Service Unavailable
C004 - Request timed out
